# Supplier Labeling Notebook

Label each supplier as **0 (reliable)** or **1 (risky/middleman)**.

Use this heuristic:
- **0 (reliable):** High shipment count, multiple buyers, years of history, valid certs
- **1 (risky):** Few shipments, one buyer, newly active, no certs, wide HS code spread

Aim for 50+ labels. Even 30 will give you a working v1 model.

In [ ]:
import duckdb
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

con = duckdb.connect('data/trust_engine.duckdb')

df = con.execute("""
    SELECT
        s.id,
        s.name,
        s.country,
        s.shipment_count,
        s.avg_monthly_shipments,
        s.total_buyers,
        s.hs_codes,
        s.first_shipment_date,
        s.last_shipment_date,
        COUNT(c.id) FILTER (WHERE c.status = 'valid') as valid_certs
    FROM suppliers s
    LEFT JOIN certifications c ON c.supplier_id = s.id
    GROUP BY s.id, s.name, s.country, s.shipment_count,
             s.avg_monthly_shipments, s.total_buyers,
             s.hs_codes, s.first_shipment_date, s.last_shipment_date
    ORDER BY s.shipment_count DESC
""").df()

print(f'Suppliers to label: {len(df)}')
df.head(20)

In [ ]:
# Manual labeling — edit this dict with your assessments
# Format: { 'supplier-id': risk_label }  where 0=reliable, 1=risky

labels = {
    # Examples — replace with real IDs from df above:
    # 'welspun-india': 0,       # Large manufacturer, well-known, high shipment count
    # 'xyz-textile-ltd': 1,     # Only 2 shipments, one buyer, no certs
}

label_df = pd.DataFrame([
    {'id': k, 'risk_label': v} for k, v in labels.items()
])

label_df.to_csv('data/labeled_suppliers.csv', index=False)
print(f'Saved {len(label_df)} labels to data/labeled_suppliers.csv')
label_df['risk_label'].value_counts()

In [ ]:
# Quick sanity check: show the most suspicious (potential 1s)
suspicious = df[
    (df['total_buyers'] <= 2) |
    (df['shipment_count'] < 5) |
    (df['valid_certs'] == 0)
].sort_values('shipment_count')

print('Likely risky suppliers (1):')
suspicious[['id', 'name', 'country', 'shipment_count', 'total_buyers', 'valid_certs']].head(20)